# Build in-silico ligand mixtures and score them with the inference model

The SIG13 inference model (`analysis/inference_model_final`) scores 38 ligand activities,
27 of them combinatorial. The calibration (`04_calibration_model_testing.Rmd`) showed the
model picks the right activity for each SIG14/SIG26 stimulation. It did not show that a
combinatorial activity means *"these cells saw A and B together"* rather than *"this profile
contains A signal and B signal"*. A bulk profile pooled from A-only and B-only cells also
contains both.

To test this, we build synthetic mixtures of A-only and B-only wells. SIG14 and SIG26 are
plate-based bulk (one well = one sample), so a mixed cell population is simply a weighted
sum of well-level count profiles.

Sections:

1. Map each stimulation to the model activity it should activate
2. Load raw UMI counts and metadata for SIG26-6h (human) and SIG14 (mouse)
3. Build `mix_AB` = 0.5 x A-alone + 0.5 x B-alone from raw counts
4. Score real and synthetic samples together, and save outputs
5. QC: compare real-well scores to the published calibration

## Setup

In [1]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import scanpy as sc

warnings.filterwarnings("ignore")

# ---- inputs -----------------------------------------------------------------------------------
# Repo-relative paths (inputs: imports_stable/, outputs: analysis_outs/)
from pathlib import Path
_here = Path.cwd().resolve()
REPO_DIR = str(next(p for p in [_here, *_here.parents] if (p / "imports_stable").is_dir()))
IMPORTS_DIR = f"{REPO_DIR}/imports_stable"
OUTS_DIR = f"{REPO_DIR}/analysis_outs/03_activity_inference_model"
SIG14_PROC = f"{IMPORTS_DIR}/SIG14/processing_outs"
SIG26_PROC = f"{IMPORTS_DIR}/SIG26/processing_outs"
CALIBRATION_DIR = f"{IMPORTS_DIR}/SIG13/analysis_outs/inference_model_calibration"
OUTDIR = f"{OUTS_DIR}/inference_model_mixture_validation"
os.makedirs(OUTDIR, exist_ok=True)

# The model itself. `model_core.py` is the seeded, single-definition port of the winning
# calibration variant (`ridge_zscore_50_weighted`); importing it rather than re-implementing
# guarantees this validation scores against exactly the deployed model.
# Shared across the activity inference analyses, in ../model_core.
sys.path.insert(0, f"{REPO_DIR}/analysis/03_activity_inference_model/model_core")
import model_core as mc

# ---- analysis constants -------------------------------------------------------------------
NONE = "none"          # the unstimulated slot label in both datasets' metadata
MIX_W = 0.5            # fraction of cells contributed by ligand A in mix_AB
CALIBRATION_MODEL = "ridge_zscore_50_weighted"

DATASETS = {
    # dataset: (count matrix, feature names, mouse -> human ortholog conversion)
    "SIG26-6h": (f"{SIG26_PROC}/count_matrix_umiDeDup_SIG26-6h.csv",
                 f"{SIG26_PROC}/featureNames_SIG26-6h.csv", True),
    "SIG14":    (f"{SIG14_PROC}/count_matrix_umiDeDup_SIG14.csv",
                 f"{SIG14_PROC}/featureNames_SIG14.csv", False),
}

## 1. Ground truth: which model activity each stimulation should activate

Each ligand pair is mapped to its expected single activities (A, B) and combinatorial
activity (AB). The mapping is copied from the `mapping_tibble` blocks in
`inference_model_final/04_calibration_model_testing.Rmd` (lines 42-55 for SIG14, 285-302
for SIG26), and extended for the CCL19/CCL21A/IL1B pairs using
`inference_model_activity_lookup.csv`.

Notes:

- Two quirks are kept from the Rmd so results stay comparable to the calibration: in SIG26,
  `IFNG` maps to `IL27_c`/`IL27_TNF_c` (not `IFNG_c`), and `IL6` maps to `IL21_c` in both
  datasets (the shared cluster is named for IL21).
- `IL1B` has no single activity in the model, so `activity_B` is `None` for the two IL1B
  pairs. They are still used for the combinatorial readout.
- SIG14's `IL6_CCL19`, `IL6_CCL21A`, `TGFb_CCL19` and `TGFb_CCL21A` are excluded because the
  model has no combinatorial activity for them (they failed the gates in
  `01_explanatory_matrix_construction.ipynb`).

In [2]:
# (ligand_A, ligand_B, activity_A, activity_B, activity_AB)
TRIPLETS = {
    "SIG26-6h": [
        ("IL2",  "TNF",    "IL2_c",  "TNFSF18_c", "IL2_TNFSF18_c"),
        ("IL4",  "TNF",    "IL4_c",  "TNFSF18_c", "IL4_TNFSF18_c"),
        ("IL6",  "TNF",    "IL21_c", "TNFSF18_c", "IL6_TNFSF18_c"),
        ("IL21", "TNF",    "IL21_c", "TNFSF18_c", "IL6_TNFSF18_c"),
        ("IL27", "TNF",    "IL27_c", "TNFSF18_c", "IL27_TNF_c"),
        ("IFNG", "TNF",    "IL27_c", "TNFSF18_c", "IL27_TNF_c"),
        ("TGFB", "TNF",    "TGFB1",  "TNFSF18_c", "TGFB1_TNFSF18_c"),
        ("IL4",  "IL21",   "IL4_c",  "IL21_c",    "IL4_IL21_c"),
        ("IL27", "IL21",   "IL27_c", "IL21_c",    "IL27_IL21_c"),
        ("TGFB", "IL4",    "TGFB1",  "IL4_c",     "TGFB1_IL4_c"),
        ("TGFB", "IL2",    "TGFB1",  "IL2_c",     "TGFB1_IL4_c"),
    ],
    "SIG14": [
        ("IL2",  "TNF",    "IL2_c",  "TNFSF18_c", "IL2_TNFSF18_c"),
        ("IL4",  "TNF",    "IL4_c",  "TNFSF18_c", "IL4_TNFSF18_c"),
        ("IL6",  "TNF",    "IL21_c", "TNFSF18_c", "IL6_TNFSF18_c"),
        ("IL21", "TNF",    "IL21_c", "TNFSF18_c", "IL6_TNFSF18_c"),
        ("IL27", "TNF",    "IL27_c", "TNFSF18_c", "IL27_TNF_c"),
        ("IFNg", "TNF",    "IFNG_c", "TNFSF18_c", "IL27_TNF_c"),
        ("TGFb", "TNF",    "TGFB1",  "TNFSF18_c", "TGFB1_TNFSF18_c"),
        ("IL21", "CCL19",  "IL21_c", "CCL19_c",   "IL21_CCL21A_c"),
        ("IL21", "CCL21A", "IL21_c", "CCL19_c",   "IL21_CCL21A_c"),
        ("IL6",  "IL1B",   "IL21_c", None,        "IL21_IL1B_c"),
        ("IL21", "IL1B",   "IL21_c", None,        "IL21_IL1B_c"),
    ],
}

X = mc.load_explanatory_matrix()
print(f"explanatory matrix X: {X.shape[0]} components x {X.shape[1]} activities")

for dataset, triplets in TRIPLETS.items():
    for triplet in triplets:
        for activity in triplet[2:]:
            assert activity is None or activity in X.columns, f"{dataset} {triplet}: {activity} not in X"
    print(f"{dataset}: {len(triplets)} triplets, all mapped activities present in X")

explanatory matrix X: 68 components x 38 activities
SIG26-6h: 11 triplets, all mapped activities present in X
SIG14: 11 triplets, all mapped activities present in X


## 2. Load counts and metadata

Counts are loaded as raw UMIs (genes x samples) so that mixing happens before any
normalization. Gene symbols are mapped the same way as in
`inference_model_final/02_calibration_input_scoring.py::_load_normalized_counts`.

Wells are identified by the *set* of ligands they received, not by slot position, because
slot order isn't consistent. For example, SIG26 has `TGFB_IL2` as the pair but `IL2_none` as
the single, with IL2 in slot 1 both times. `stim_key` handles this.

In [3]:
def stim_key(*ligands):
    """Slot-independent key for a well's stimulation. Control -> '', single -> 'IL2',
    pair -> 'IL2|TNF' (alphabetical)."""
    return "|".join(sorted(ligand for ligand in ligands if ligand != NONE))


def load_counts(counts_path, features_path):
    """Raw UMI counts as genes (symbols) x samples."""
    counts = pd.read_csv(counts_path, index_col=0)
    features = pd.read_csv(features_path, index_col=0)
    counts = counts.loc[counts.index.isin(features.index), :]
    counts = counts.merge(features, left_index=True, right_index=True)
    return counts.set_index("gene").drop(columns=["category"])


def load_metadata(dataset, sample_ids):
    """Per-sample ligand/replicate annotation, joined on sample ID (never positional)."""
    if dataset == "SIG14":
        # SIG14 has no condition column; sample IDs encode everything
        meta = pd.DataFrame({"sample_id": sample_ids})
        meta[["ligand1", "ligand2", "replicate", "well", "library", "project"]] = \
            meta["sample_id"].str.split("_", expand=True)
    else:
        meta = (pd.read_csv(f"{SIG26_PROC}/processed_metadata_{dataset}.csv")
                  .rename(columns={"sample_ID": "sample_id"})
                  [["sample_id", "ligand1", "ligand2", "replicate", "well"]])
        meta["library"] = dataset
    assert set(meta.sample_id) == set(sample_ids), "metadata and count matrix disagree on samples"
    meta["stim"] = [stim_key(a, b) for a, b in zip(meta.ligand1, meta.ligand2)]
    return meta


def well_index(meta):
    """(stim_key, replicate) -> sample_id. SIG26 has two `none_none` and two `none_TNF`
    wells per donor; take the first by well position so the choice is deterministic."""
    return (meta.sort_values("well")
                .groupby(["stim", "replicate"])["sample_id"].first().to_dict())

## 3. Build mixtures

A bulk profile of a mixed cell population is the cell-weighted average of the two
populations in linear (count) space, not log space. So each parent well is converted to
fractions of its own library, the two are averaged, and the result is rescaled to the mean
of the two parents' sequencing depths.

Mixtures are appended to the raw count matrix, so they go through exactly the same
`normalize_total` -> `log1p` -> `waggr` -> ridge pipeline as the real wells.

Each ligand's signal is diluted by half in the mixture. This is intended: it matches a real
co-occurrence, where only some cells see A and the rest see B.

In [4]:
def mix_counts(counts, sample_a, sample_b, w=MIX_W):
    """Synthetic bulk profile of a population that is `w` cells from `sample_a` and
    `1 - w` from `sample_b`."""
    a = counts[sample_a] / counts[sample_a].sum()
    b = counts[sample_b] / counts[sample_b].sum()
    depth = 0.5 * (counts[sample_a].sum() + counts[sample_b].sum())
    return (w * a + (1 - w) * b) * depth


def build_mixtures(dataset, counts, meta):
    """For every triplet x replicate with all four real wells present, emit the synthetic
    `mix_AB` profile and a long index row per sample role.

    Real wells are shared across triplets (every TNF pair reuses the same `none_TNF` well),
    so the index is long -- one row per triplet x replicate x role -- while each real well is
    scored only once.
    """
    index_rows, mix_cols, skipped = [], {}, []
    for ligand_a, ligand_b, activity_a, activity_b, activity_ab in TRIPLETS[dataset]:
        for replicate in sorted(meta.replicate.unique()):
            roles = {"real_control":  stim_key(),
                     "real_single_A": stim_key(ligand_a),
                     "real_single_B": stim_key(ligand_b),
                     "real_combo":    stim_key(ligand_a, ligand_b)}
            wells = {role: index.get((key, replicate)) for role, key in roles.items()}
            if any(sample is None for sample in wells.values()):
                skipped.append(f"{ligand_a}+{ligand_b} / {replicate}")
                continue

            mix_id = f"MIX__{ligand_a}__{ligand_b}__{replicate}"
            mix_cols[mix_id] = mix_counts(counts, wells["real_single_A"], wells["real_single_B"])
            wells["mix_AB"] = mix_id

            for role, sample_id in wells.items():
                index_rows.append(dict(
                    dataset=dataset, triplet=f"{ligand_a}+{ligand_b}",
                    ligand_A=ligand_a, ligand_B=ligand_b, replicate=replicate,
                    sample_type=role, sample_id=sample_id,
                    activity_A=activity_a, activity_B=activity_b, activity_AB=activity_ab))

    return pd.DataFrame(index_rows), mix_cols, skipped

## 4. Score real and synthetic samples together

Each dataset **must** be scored in a single run. `mc.score_components` calls `sc.pp.scale`,
which z-scores each gene across all samples, so real and synthetic samples need to share the
same scaling. Setting `sample_keys=["sample_id"]` keeps each well as its own sample.

For mouse data, `convert_to_human=True` uses the saved `mouse_human_ortholog_map.csv` in
`../model_core/` (covers all 5,045 genes in the component net), so no
BioMart query is needed.

In [5]:
for dataset, (counts_path, features_path, to_human) in DATASETS.items():
    print(f"\n{'=' * 20} {dataset} {'=' * 20}", flush=True)

    counts = load_counts(counts_path, features_path)
    meta = load_metadata(dataset, list(counts.columns))
    index = well_index(meta)
    print(f"counts: {counts.shape[0]} genes x {counts.shape[1]} wells", flush=True)

    sample_index, mix_cols, skipped = build_mixtures(dataset, counts, meta)
    counts_all = pd.concat([counts, pd.DataFrame(mix_cols, index=counts.index)], axis=1)
    used = sorted(set(sample_index.sample_id))

    # A synthetic profile's depth must be the mean of its two parents' depths.
    for _, row in sample_index.query("sample_type == 'mix_AB'").iterrows():
        parents = sample_index.query("triplet == @row.triplet and replicate == @row.replicate")
        a = parents.query("sample_type == 'real_single_A'").sample_id.iloc[0]
        b = parents.query("sample_type == 'real_single_B'").sample_id.iloc[0]
        assert np.isclose(counts_all[row.sample_id].sum(), 0.5 * (counts[a].sum() + counts[b].sum()))

    print(f"{sample_index.triplet.nunique()} triplets -> "
          f"{len(mix_cols)} triplet x replicate mixtures, "
          f"{len(used)} unique samples to score", flush=True)
    if skipped:
        # Expected: SIG14 lib1 and lib2 were separate mouse cohorts, so their triplets do not
        # resolve against each other's replicates.
        print(f"skipped {len(skipped)} triplet x replicate combinations with missing wells "
              f"(e.g. {skipped[0]})", flush=True)

    rna = sc.AnnData(counts_all[used].T.astype(np.float32))
    rna.var_names_make_unique()
    sc.pp.normalize_total(rna)
    sc.pp.log1p(rna)
    rna.layers["log1p_norm"] = rna.X.copy()
    rna.obs["sample_id"] = rna.obs_names

    net, scoring_genes = mc.build_spca_net(convert_to_human=to_human)
    obs = mc.score_components(rna, net, scoring_genes)
    Y = mc.build_target_matrix(obs, sample_keys=["sample_id"], component_order=X.index)
    activity, r2 = mc.score_ligand_activity(X, Y)
    print(f"activity: {activity.shape}, median R2 = {r2.r2_score.median():.3f}", flush=True)

    activity.to_csv(f"{OUTDIR}/activity_mixtures_{dataset}.csv", index=False)
    r2.to_csv(f"{OUTDIR}/r2_mixtures_{dataset}.csv", index=False)
    sample_index.to_csv(f"{OUTDIR}/sample_index_{dataset}.csv", index=False)
    print(f"saved activity_mixtures_{dataset}.csv, r2_mixtures_{dataset}.csv, "
          f"sample_index_{dataset}.csv", flush=True)


==================== SIG26-6h ====================


counts: 41896 genes x 66 wells


11 triplets -> 33 triplet x replicate mixtures, 93 unique samples to score


  scoring genes: kept 1454/1454 (detected in >=1% of 93 cells)


activity: (93, 39), median R2 = 0.691


...

11 triplets -> 42 triplet x replicate mixtures, 140 unique samples to score


skipped 35 triplet x replicate combinations with missing wells (e.g. IL2+TNF / mouse3)


  scoring genes: kept 1512/1512 (detected in >=1% of 140 cells)


activity: (140, 39), median R2 = 0.932


saved activity_mixtures_SIG14.csv, r2_mixtures_SIG14.csv, sample_index_SIG14.csv, components_mixtures_SIG14.csv


## 5. QC: do the real wells match the published calibration?

The real wells here were also scored in step 02 of `inference_model_final`, so their
activity scores should agree closely, but not exactly. Differences come from:

- a different alpha grid: `logspace(-1, 4, 500)` here vs. `logspace(-1, 3, 100)` in step 02
- a seeded permutation null here vs. unseeded in step 02
- the added synthetic samples, which shift `sc.pp.scale`

So we report correlations instead of checking for equality.

In [6]:
for dataset in DATASETS:
    activity = pd.read_csv(f"{OUTDIR}/activity_mixtures_{dataset}.csv")
    activities = [c for c in activity.columns if c != "sample"]

    published = (pd.read_csv(f"{CALIBRATION_DIR}/activity_combined_{dataset}.csv")
                   .query("model == @CALIBRATION_MODEL"))

    real = activity[~activity["sample"].str.startswith("MIX__")].copy()
    if dataset == "SIG14":   # sample_id is lig1_lig2_rep_well_lib_project; published key drops lib/project
        real["key"] = real["sample"].str.split("_").str[:4].str.join("_")
    else:                    # sample_id is condition_lig1_lig2_rep_well; published key drops condition
        real["key"] = real["sample"].str.split("_").str[1:].str.join("_")

    merged = (real.melt(id_vars="key", value_vars=activities, var_name="activity", value_name="z_new")
                  .merge(published.melt(id_vars="sample", value_vars=activities,
                                        var_name="activity", value_name="z_published")
                                  .rename(columns={"sample": "key"}),
                         on=["key", "activity"], how="inner"))
    assert merged.key.nunique() == real.key.nunique(), "some real wells did not match the published run"
    print(f"{dataset}: {merged.key.nunique()} real wells matched, "
          f"pearson r = {merged.z_new.corr(merged.z_published):.3f}, "
          f"spearman = {merged.z_new.corr(merged.z_published, method='spearman'):.3f}")

SIG26-6h: 60 real wells matched, pearson r = 0.992, spearman = 0.991
SIG14: 98 real wells matched, pearson r = 0.952, spearman = 0.945
